In [ ]:
import numpy as np
import time, os, sys, random
from urllib.parse import urlparse
import skimage.io
from skimage import measure
import matplotlib.pyplot as plt
import matplotlib as mpl
skimage.io.use_plugin('matplotlib')
from glob import glob
import pandas as pd
from statistics import mode
import collections
import pickle
import seaborn as sns
%matplotlib inline
mpl.rcParams['figure.dpi'] = 200
mpl.rcParams.update({'font.size': 6})

import zlib
import base64
from PIL import Image
from sklearn.linear_model import LinearRegression

# Experiment 1: Nucleolar area

In [ ]:
data_dir1 = '021723_F1-E15.25-nucleolin-total-area-AC'
data_dir2 = '021923_F1-E15.25-nucleolin-number-AC'
data_dir3 = '022223_F1-E15.25-nucleolin-total-area-second-plate-D'
data_dir4 = '022323_F1-E15.25-nucleolin-total-area-first-plate-DC6'
data_dir5 = '022423_F1-E15.25-nucleolin-number-third-plate-AB'
data_dir6 = '022623_F1-E15.25-nucleolin-total-area-third-plate-CB'

data_dirs = [data_dir1, data_dir2, data_dir3, data_dir4, data_dir5, data_dir6]
data_dir_area = [data_dir1, data_dir3, data_dir4, data_dir6]
data_dir_number = [data_dir2, data_dir5]

In [ ]:
cells_df1 = pd.read_csv(data_dir1 + '/cells.csv').dropna()
cells_df2 = pd.read_csv(data_dir3 + '/cells.csv').dropna()
cells_df3 = pd.read_csv(data_dir4 + '/cells.csv').dropna()
cells_df4 = pd.read_csv(data_dir6 + '/cells.csv').dropna()

In [ ]:
combined_df = pd.concat([cells_df1, cells_df3, cells_df4], ignore_index=True)
filtered_combined_df = combined_df[combined_df['Nucleoli_Called'] == 2]

q1, median, q3 = filtered_combined_df['Nucleolar_Area'].quantile([0.25, 0.5, 0.75])

plt.figure(figsize=(6, 6), dpi=300)
ax = sns.kdeplot(filtered_combined_df['Nucleolar_Area'], fill=True, color='#3498db', alpha=0.4, linewidth=2)

quantiles = [q1, median, q3]
labels = [f'Q1: {q1:.0f}', f'Q2: {median:.0f}', f'Q3: {q3:.0f}']

y_max = ax.get_ylim()[1]
heights = [y_max * 0.92, y_max * 0.82, y_max * 0.72]

for q, label, h in zip(quantiles, labels, heights):
    plt.axvline(q, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    plt.text(q + 50, h, label, color='red', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='none', pad=1))

plt.xlim([0, 5000])

plt.title('Experiment 1: Nucleolar area', fontsize=18, pad=10)
plt.xlabel('Absolute nucleolar area', fontsize=15)
plt.ylabel('Density', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Experiment 2: Nucleolar and SiRDNA area

In [ ]:
data_dir1 = '040223_F1-E15.25-SiRDNA-plate1-total-area/'
data_dir2 = '040323_F1-E15.25-SiRDNA-plate1-total-area-B/'
data_dir3 = '040523_F1-E15.25-nucleolin-plate2-total-area/'
data_dir4 = '040623_F1-E15.25-SiRDNA-plate2-total-area/'
data_dir5 = '040723_F1-E15.25-SiRDNA-plate2-C-total-area/'
data_dir6 = '040823_F1-E15.25-nucleolin-plate3-total-area/'
data_dir7 = '040923_F1-E15.25-SiRDNA-plate3-total-area/'
data_dir8 = '041023_F1-E15.25-nucleolin-plate4-total-area-A/'
data_dir9 = '041123_F1-E15.25-SiRDNA-plate4-total-area/'
data_dir10 = '041223_F1-E15.25-nucleolin-plate5-total-area/'
data_dir11 = '041323_F1-E15.25-SiRDNA-plate5-total-area/'
data_dir12 = '041423_F1-E15.25-nucleolin-plate6-total-area/'
data_dir13 = '041623_F1-E15.25-SiRDNA-plate6-total-area/'
data_dir14 = '041623_F1-E15.25-nucleolin-plate4-B-total-area/'
data_dir15 = '041623_F1-E15.25-nucleolin-plate5-B-total-area/'
data_dir16 = '041823_F1-E15.25-nucleolin-plate5-B04-total-area/'

data_dir_sir = [data_dir1, data_dir2, data_dir4, data_dir5, data_dir7, data_dir9, data_dir11, data_dir13]
data_dir_nucl = [data_dir3, data_dir6, data_dir8, data_dir10, data_dir12, data_dir14, data_dir15, data_dir16]

In [ ]:
cell_dfs_sir = {}
cell_dfs_nucl = {}

for d in data_dir_sir:
    cells_df = pd.read_csv(d + 'cells.csv').dropna()
    cell_dfs_sir[d] = cells_df
    
for d in data_dir_nucl:
    cells_df = pd.read_csv(d + 'cells.csv').dropna()
    cell_dfs_nucl[d] = cells_df

In [ ]:
combined_df = pd.concat(cell_dfs_nucl.values(), ignore_index=True)
filtered_combined_df = combined_df[combined_df['Nucleoli_Called'] == 2]

In [ ]:
q1, median, q3 = filtered_combined_df['Nucleolar_Area'].quantile([0.25, 0.5, 0.75])

plt.figure(figsize=(6, 6), dpi=300)
ax = sns.kdeplot(filtered_combined_df['Nucleolar_Area'], fill=True, color='#3498db', alpha=0.4, linewidth=2)

quantiles = [q1, median, q3]
labels = [f'Q1: {q1:.0f}', f'Q2: {median:.0f}', f'Q3: {q3:.0f}']

y_max = ax.get_ylim()[1]
heights = [y_max * 0.92, y_max * 0.82, y_max * 0.72]

for q, label, h in zip(quantiles, labels, heights):
    plt.axvline(q, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    plt.text(q + 50, h, label, color='red', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='none', pad=1))

plt.xlim([0, 5000])

plt.title('Experiment 2: Nucleolar area', fontsize=18, pad=10)
plt.xlabel('Absolute nucleolar area', fontsize=15)
plt.ylabel('Density', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Experiment 3: SC35 area

In [ ]:
data_dir1 = '041823_F1-E15.25-SC35-plate1-total-area/'
data_dir2 = '041923_F1-E15.25-SC35-plate1-ratio/'
data_dir3 = '042023_F1-E15.25-SC35-plate1C-total-area/'
data_dir4 = '042123_F1-E15.25-SC35-plate2-total-area/'
data_dir5 = '042223_F1-E15.25-SC35-plate2-ratio/'
data_dir6 = '042423_F1-E15.25-SC35-plate3-ratio/'
data_dir7 = '042523_F1-E15.25-SC35-plate4-total-area/'
data_dir8 = '042623_F1-E15.25-SC35-plate4-ratio/'
data_dir9 = '042723_F1-E15.25-SC35-plate3-total-area-C/'
data_dir10 = '042723_F1-E15.25-SC35-plate4-total-area-C/'
data_dir11 = '042823_F1-E15.25-SC35-plate4-ratio-C/'
data_dir12 = '042823_F1-E15.25-SC35-plate4-ratio-C06/'

data_dir_missing = '042323_F1-E15.25-SC35-plate3-total-area/'

data_dir_area = [data_dir1, data_dir3, data_dir4, data_dir7, data_dir9, data_dir10]
data_dir_fraction = [data_dir2, data_dir5, data_dir6, data_dir8, data_dir11, data_dir12]

In [ ]:
def make_dict(wells, RT_wells):
    tuples = [(key, value)
          for i, (key, value) in enumerate(zip(wells, RT_wells))]
    res = dict(tuples)
    return(res)

In [ ]:
RT_wells_area = {}
RT_wells_fraction = {}

RT_wells_area[data_dir1] = [289, 290, 291, 292, 293, 294, 306]
RT_wells_area[data_dir3] = [315, 316, 317, 318]
RT_wells_area[data_dir4] = [325, 326, 327, 328, 329, 330, 342, 341, 340]
RT_wells_area[data_dir7] = [331, 332, 333, 334, 335, 336]
RT_wells_area[data_dir9] = [323, 324]
RT_wells_area[data_dir10] = [355, 356, 357, 358]

RT_wells_fraction[data_dir2] = [305, 304, 303, 302, 301, 313, 314]
RT_wells_fraction[data_dir5] = [339, 338, 337, 349, 350, 351, 352, 353, 354]
RT_wells_fraction[data_dir6] = [310, 309, 308, 307, 319, 320, 321, 322]
RT_wells_fraction[data_dir8] = [348, 347, 346, 345, 344, 343]
RT_wells_fraction[data_dir11] = [359]
RT_wells_fraction[data_dir12] = [360]

In [ ]:
cell_dfs_area = {}
cell_dfs_fraction = {}

for d in data_dir_area:
    if d == data_dir1:
        cells_df = pd.read_csv(d + 'cells.csv')
        cells_df['UV_Status'] = cells_df['Nuclear_Area']
        cells_df['Nuclear_Area'] = cells_df['Nucleolar_Area'] / cells_df['Nucleolar_Ratio']
    else:
        cells_df = pd.read_csv(d + 'cells.csv').dropna()
    
    cells_df['Well'] = cells_df['ImageDir'].apply(lambda x: x.split('/')[6].split('_')[2])
    well_dict = make_dict(cells_df['Well'].unique(), RT_wells_area[d])
    cells_df['RT_well'] = cells_df['Well'].map(well_dict).astype('Int64')
    cell_dfs_area[d] = cells_df.dropna()
    
for d in data_dir_fraction:
    if d == data_dir2:
        cells_df = pd.read_csv(d + 'cells.csv')
        cells_df['UV_Status'] = cells_df['Nuclear_Area']
        cells_df['Nuclear_Area'] = cells_df['Nucleolar_Area'] / cells_df['Nucleolar_Ratio']
    else:
        cells_df = pd.read_csv(d + 'cells.csv').dropna()
        
    cells_df['Well'] = cells_df['ImageDir'].apply(lambda x: x.split('/')[6].split('_')[2])
    well_dict = make_dict(cells_df['Well'].unique(), RT_wells_fraction[d])
    cells_df['RT_well'] = cells_df['Well'].map(well_dict).astype('Int64')
    cell_dfs_fraction[d] = cells_df.dropna()

In [ ]:
combined_df = pd.concat(cell_dfs_area.values(), ignore_index=True)

In [ ]:
q1, median, q3 = combined_df['Nucleolar_Area'].quantile([0.25, 0.5, 0.75])

plt.figure(figsize=(6, 6), dpi=300)
ax = sns.kdeplot(combined_df['Nucleolar_Area'], fill=True, color='#3498db', alpha=0.4, linewidth=2)

quantiles = [q1, median, q3]
labels = [f'Q1: {q1:.0f}', f'Q2: {median:.0f}', f'Q3: {q3:.0f}']

y_max = ax.get_ylim()[1]
heights = [y_max * 0.92, y_max * 0.82, y_max * 0.72]

for q, label, h in zip(quantiles, labels, heights):
    plt.axvline(q, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    plt.text(q + 50, h, label, color='red', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='none', pad=1))

plt.xlim([0, 3500])

plt.title('Experiment 3: Speckle area', fontsize=18, pad=10)
plt.xlabel('Absolute speckle area', fontsize=15)
plt.ylabel('Density', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
combined_df = pd.concat(cell_dfs_fraction.values(), ignore_index=True)

In [ ]:
q1, median, q3 = combined_df['Nucleolar_Ratio'].quantile([0.25, 0.5, 0.75])

plt.figure(figsize=(6, 6), dpi=300)
ax = sns.kdeplot(combined_df['Nucleolar_Ratio'], fill=True, color='#3498db', alpha=0.4, linewidth=2)

quantiles = [q1, median, q3]
labels = [f'Q1: {q1:.3f}', f'Q2: {median:.3f}', f'Q3: {q3:.3f}']

y_max = ax.get_ylim()[1]
heights = [y_max * 0.92, y_max * 0.82, y_max * 0.72]

x_offset = 0.005
for q, label, h in zip(quantiles, labels, heights):
    plt.axvline(q, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    plt.text(q + x_offset, h, label, color='red', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='none', pad=1))

plt.xlim([0, 0.3])

plt.title('Experiment 3: Speckle fraction', fontsize=18, pad=10)
plt.xlabel('Speckle fraction', fontsize=15)
plt.ylabel('Density', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Experiment 4: Nuclear area & nucleolar fraction

In [ ]:
data_dir1 = '070723_F1-E15.25-nuclear-size-plate1/'
data_dir2 = '070823_F1-E15.25-nucleolar-fraction-plate1/'
data_dir3 = '070923_F1-E15.25-nuclear-size-plate2/'
data_dir4 = '071023_F1-E15.25-nucleolar-fraction-plate2/'
data_dir5 = '071223_F1-E15.25-nucleolar-fraction-plate3/'
data_dir6 = '071323_F1-E15.25-nuclear-size-plate3/'
data_dir7 = '071423_F1-E15.25-nucleolar-fraction-plate4/'
data_dir8 = '071523_F1-E15.25-nuclear-size-plate4/'
data_dir9 = '071623_F1-E15.25-nucleolar-fraction-plate5/'
data_dir10 = '071723_F1-E15.25-nuclear-size-plate5/'


data_dir_area = [data_dir1, data_dir3, data_dir6, data_dir8, data_dir10]
data_dir_fraction = [data_dir2, data_dir4, data_dir5, data_dir7, data_dir9]

In [ ]:
RT_wells_area = {}
RT_wells_fraction = {}

RT_wells_area[data_dir1] = [289, 290, 291, 292, 293, 294, 306, 305, 304]
RT_wells_area[data_dir3] = [325, 326, 327, 328, 329, 330, 342, 341, 340]
RT_wells_area[data_dir6] = [295, 296, 297, 298, 299, 300, 312, 311, 310]
RT_wells_area[data_dir8] = [331, 332, 333, 334, 335, 336, 348, 347, 346]
RT_wells_area[data_dir10] = [361, 362, 363, 364, 365, 366, 372, 371, 370]


RT_wells_fraction[data_dir2] = [303, 302, 301, 313, 314, 315, 316, 317, 318]
RT_wells_fraction[data_dir4] = [339, 338, 337, 349, 350, 351, 352, 353, 354]
RT_wells_fraction[data_dir5] = [309, 308, 307, 319, 320, 321, 322, 323, 324]
RT_wells_fraction[data_dir7] = [345, 344, 343, 355, 356, 357, 358, 359, 360]
RT_wells_fraction[data_dir9] = [369, 368, 367, 373, 374, 375, 376, 377, 378]

In [ ]:
cell_dfs_area = {}
cell_dfs_fraction = {}

for d in data_dir_area:
    cells_df = pd.read_csv(d + 'cells.csv').dropna() 
    cells_df['Well'] = cells_df['ImageDir'].apply(lambda x: x.split('/')[6].split('_')[2])
    well_dict = make_dict(cells_df['Well'].unique(), RT_wells_area[d])
    cells_df['RT_well'] = cells_df['Well'].map(well_dict)
    cell_dfs_area[d] = cells_df
    
for d in data_dir_fraction:
    cells_df = pd.read_csv(d + 'cells.csv').dropna()    
    cells_df['Well'] = cells_df['ImageDir'].apply(lambda x: x.split('/')[6].split('_')[2])
    well_dict = make_dict(cells_df['Well'].unique(), RT_wells_fraction[d])
    cells_df['RT_well'] = cells_df['Well'].map(well_dict)
    cell_dfs_fraction[d] = cells_df

In [ ]:
combined_df = pd.concat(cell_dfs_fraction.values(), ignore_index=True)
filtered_combined_df = combined_df[combined_df['Nucleoli_Called'] == 2]

In [ ]:
q1, median, q3 = filtered_combined_df['Nucleolar_Fraction'].quantile([0.25, 0.5, 0.75])

plt.figure(figsize=(6, 6), dpi=300)
ax = sns.kdeplot(filtered_combined_df['Nucleolar_Fraction'], fill=True, color='#3498db', alpha=0.4, linewidth=2)

quantiles = [q1, median, q3]
labels = [f'Q1: {q1:.3f}', f'Q2: {median:.3f}', f'Q3: {q3:.3f}']

y_max = ax.get_ylim()[1]
heights = [y_max * 0.92, y_max * 0.82, y_max * 0.72]

x_offset = 0.005
for q, label, h in zip(quantiles, labels, heights):
    plt.axvline(q, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    plt.text(q + x_offset, h, label, color='red', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='none', pad=1))

plt.xlim([0, 0.4])

plt.title('Experiment 4: Nucleolar fraction', fontsize=18, pad=10)
plt.xlabel('Nucleolar fraction', fontsize=15)
plt.ylabel('Density', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
combined_df = pd.concat(cell_dfs_area.values(), ignore_index=True)
combined_df.head()

In [ ]:
q1, median, q3 = combined_df['Nuclear_Area'].quantile([0.25, 0.5, 0.75])

plt.figure(figsize=(6, 6), dpi=300)
ax = sns.kdeplot(combined_df['Nuclear_Area'], fill=True, color='#3498db', alpha=0.4, linewidth=2)

quantiles = [q1, median, q3]
labels = [f'Q1: {q1:.0f}', f'Q2: {median:.0f}', f'Q3: {q3:.0f}']

y_max = ax.get_ylim()[1]
heights = [y_max * 0.92, y_max * 0.82, y_max * 0.72]

for q, label, h in zip(quantiles, labels, heights):
    plt.axvline(q, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    plt.text(q + 50, h, label, color='red', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='none', pad=1))

plt.xlim([3500, 17000])

plt.title('Experiment 4: Nuclear area', fontsize=18, pad=10)
plt.xlabel('Nuclear area', fontsize=15)
plt.ylabel('Density', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()